# 2025-03-10: Distribution of the Transitive and Intransitive Verbs

## Prelude

In [1]:
%load_ext autoreload
%autoreload 2

In [43]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
from os import environ
import apfel as apfel
import statsmodels.api as sm

In [4]:
dataset = pl.read_parquet(f"{environ['DATA_PATH_BASE']}/v2/babylm/0310/parsed-roberta.pos-lemma.parquet")

In [5]:
dataset

index,text,split,parse,lemma,pos
u32,str,enum,struct[3],list[str],list[str]
0,"""aboo .""","""train""",null,null,null
1,"""kee .""","""train""",null,null,null
2,"""hey .""","""train""",null,null,null
3,"""smile ?""","""train""",null,null,null
4,"""hm hm .""","""train""","{[""root"", ""dep"", ""punct""],[0, 1, 1],[""hm"", ""hm"", "".""]}","[""hm"", ""hm"", "".""]","[""INTJ"", ""INTJ"", ""PUNCT""]"
…,…,…,…,…,…
209995,"""oh my heavens ?""","""test""","{[""root"", ""poss"", … ""punct""],[0, 3, … 1],[""oh"", ""my"", … ""?""]}","[""oh"", ""my"", … ""?""]","[""INTJ"", ""PRON"", … ""PUNCT""]"
209996,"""oh hi jessica .""","""test""","{[""root"", ""dep"", … ""punct""],[0, 1, … 3],[""oh"", ""hi"", … "".""]}","[""oh"", ""hi"", … "".""]","[""INTJ"", ""INTJ"", … ""PUNCT""]"
209997,"""are you helping the babysitter…","""test""","{[""aux"", ""nsubj"", … ""punct""],[3, 3, … 3],[""are"", ""you"", … ""?""]}","[""be"", ""you"", … ""?""]","[""AUX"", ""PRON"", … ""PUNCT""]"


## Extract Transitive Verbs

In [6]:
from functools import partial
from verb_overgen.prep.lib.pattern_matching import match_svo

dataset_trans_verb = dataset.filter(
    pl.col("parse").is_not_null(),
    pl.col("parse").struct.field("rel").pipe(lambda r: r.list.contains("nsubj") & r.list.contains("dobj"))
).select(
    pl.col("index"),
    trans_verb_index := pl.col("parse").pipe(match_svo).alias("trans_verb_index"),
    trans_verb := pl.col("lemma").list.gather(trans_verb_index - 1).alias("trans_verb"),
).filter(
    pl.col("trans_verb_index").list.len() > 0
)

In [7]:
df_all_trans_verb = dataset_trans_verb.select(
    pl.col("trans_verb").explode().value_counts().struct.unnest()
).sort(pl.col("count"), pl.col("trans_verb"), descending=[True, False])

df_all_trans_verb

trans_verb,count
str,u32
"""have""",5652
"""do""",4553
"""put""",2623
"""get""",2518
"""see""",2058
…,…
"""wolfs""",1
"""worth""",1
"""yank""",1


node

node <- subj <- (*)
node <- dobj <- (*)

node <=> transitive verb

## Extract Verbs

### Inspect spaCy's POS Tags

In [8]:
with pl.Config(set_tbl_rows=-1):
    display(dataset.select(
        pl.col("pos").explode().value_counts().struct.unnest()
    ))

pos,count
str,u32
"""NOUN""",152223
"""VERB""",163888
"""X""",1644
null,38926
"""PRON""",179999
"""NUM""",8481
"""ADJ""",40199
"""ADV""",64248
"""PART""",17645


### Extract Verbs

In [9]:
df_all_verb = dataset.select(
    imperative(
        verb_index := pl.col("pos").list.eval(pl.element().eq("VERB").arg_true()),
        pl.col("lemma").list.gather(verb_index).explode()
    ).alias("verb")
).filter(
    pl.col("verb").is_not_null()
).select(
    pl.col("verb").value_counts().struct.unnest()
).sort(
    pl.col("count"), pl.col("verb"), descending=[True, False]
)

df_all_verb

verb,count
str,u32
"""go""",10615
"""do""",9012
"""have""",8630
"""get""",6681
"""see""",6648
…,…
"""zigzag""",1
"""zoo""",1
"""zoop""",1


### POS Tagging -- Dep Parsing Cross Validation

Should the parsing be accurate, a verb's appearance should be greater than it's transitive appearance. There are 9 "verbs" that violate this rule.

In [10]:
df_all_verb_2 = df_all_verb.join(
    df_all_trans_verb.rename({"trans_verb": "verb", "count": "count_trans"}),
    on="verb",
    how="full",
    coalesce=True,
)

df_all_verb_2.filter(
    pl.col("count").lt(pl.col("count_trans"))
)

verb,count,count_trans
str,u32,u32
"""be""",723,895
"""empty""",6,9
"""loose""",6,10
"""buckle""",3,5
"""givin""",2,3
"""bag""",1,2
"""can""",1,2
"""interest""",1,2
"""took""",1,2


There are also few transitive verbs that are not tagged as verbs by the POS tagger.

In [11]:
df_all_verb_2.filter(
    pl.col("count").is_null()
).select(
    pl.col("count_trans").count().alias("count_pos_tagging_error"),
    pl.sum("count_trans").alias("count_pos_tagging_error_total"),
)

count_pos_tagging_error,count_pos_tagging_error_total
u32,u32
66,87


In [14]:
df_trans_intrans = df_all_verb_2.fill_null(0).with_columns(
    pl.when(
        pl.col("count").ge(pl.col("count_trans")),
    ).then(
        pl.col("count").sub(pl.col("count_trans"))
    ).otherwise(
        pl.lit(0)
    ).alias("count_intrans")
).filter(
    pl.col("count_intrans").gt(0)
).sort(
    pl.col("count_trans"), pl.col("count_intrans"), pl.col("verb"), descending=[True, False, False]
)

fig, ax = plt.subplots(figsize=(10, 10))

sns.scatterplot(
    df_trans_intrans,
    ax=ax,
    x="count_trans", y="count_intrans",
    hue="count",
    palette="mako_r",
    marker="X",
)

for (verb, count, count_trans, count_intrans) in df_trans_intrans.filter(pl.col("count").gt(1000)).iter_rows():
    ax.text(
        count_trans, count_intrans,
        f"{verb}",
        fontsize=8,
    )


ax.set_xscale("log")
ax.set_yscale("log")
ax.set_aspect("equal")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Total Count")
plt.savefig("trans-intrans.png")

![](./trans-intrans.png)

# Bidgood (2021) Verbs Revisited

In [20]:
from verb_overgen.prep.lib.lexical import BIDGOOD_VERBS

df_bidgood_v = pl.DataFrame([(category, word) for category, s in BIDGOOD_VERBS.items() for word in s], orient="row", schema=pl.Schema({
    "category": pl.Enum([*BIDGOOD_VERBS.keys(), "non_bidgood"]),
    "word": pl.String
}))

In [ ]:
import numpy as np


df_trans_intrans_bidgood = df_trans_intrans.join(df_bidgood_v, left_on="verb", right_on="word", how="inner", coalesce=True).with_columns(
    pl.col("count", "count_trans", "count_intrans").fill_null(0),
    # pl.col("category").fill_null("non_bidgood"),
)

fig, ax = plt.subplots(figsize=(10, 10))

sns.scatterplot(
    df_trans_intrans_bidgood,
    ax=ax,
    x="count_trans", y="count_intrans",
    hue="category",
    marker="X",
)

for (verb, _, count_trans, count_intrans, *_) in df_trans_intrans_bidgood.iter_rows():
    ax.text(
        count_trans, count_intrans,
        f"{verb}",
        fontsize=8,
    )

df_trans_intrans_bidgood_linear = df_trans_intrans_bidgood.group_by("category", maintain_order=True).agg(
    pl.map_groups([pl.col("count_trans"), pl.col("count_intrans")], lambda l: imperative(
        sm.OLS(l[1].to_numpy(), sm.add_constant(l[0].to_numpy())).fit().params),
    ).alias("param"))

for idx, (category, param) in enumerate(df_trans_intrans_bidgood_linear.iter_rows()):
    sns.lineplot(
        ax=ax,
        x=np.arange(1, 1e4),
        y=param[0] + param[1] * np.arange(1, 1e4),
        label=category,
        color=sns.color_palette()[idx],
        legend=False,
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_aspect("equal")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Category")

plt.savefig("trans-intrans-bidgood.png")

Bidgood (2021) is almost accurate in categorizing the verbs.
However, there are too few transitive verbs that are completely not intransitive.

![](./trans-intrans-bidgood.png)

Some of the "intransitive" cases are actually not verbs or not intransitive.
Should work on a better way to extract pure intransitive verbs.

In [132]:
case = "wait"

In [ ]:
from rich.table import Table

table = Table(
    "index", "text", "is_transitive",
    title=f"Case Study on '{case}'"
)

df = dataset.filter(
    pl.col("lemma").list.contains(case)
).select(
    pl.col("index"),
    pl.col("text"),
    pl.col("index").is_in(dataset_trans_verb["index"]).alias("is_trans")
)

for idx, text, is_trans in df.iter_rows():
    text = text.replace(f"{case}", f"[u b]{case}[/]")
    is_trans = "[blue]transitive[/]" if is_trans else "[yellow]intransitive[/]"
    table.add_row(str(idx), text, str(is_trans))

display(table)

In [136]:
idx = 10763

In [ ]:
from verb_overgen.prep.lib.graph import draw_parse_graph, create_parse_graph

graph = create_parse_graph(dataset[idx]["parse"].item());

for (i, data) in graph.nodes.data():
    if i > 0:
        data["label"] += f"\n{dataset[idx]['pos'].item()[i - 1]}"

draw_parse_graph(graph);